# 5주 2차시 · 조건으로 고르기와 정렬하기

**학습 목표**
1. `df['열']`(Series)과 `df[['열1', '열2']]`(DataFrame)의 차이를 안다.
2. True/False 조건으로 원하는 행만 고른다.
3. `&`(그리고)·`|`(또는)·`isin`으로 조건을 묶고, 결과의 개수와 비율을 센다.
4. `sort_values`·`nlargest`·`nsmallest`로 상위·하위 행을 찾는다.

**사용 데이터**: `process.csv` — 사출 성형 공정 기록 494행(가상 데이터)

오늘은 현장에서 나온 **질문 다섯 개**에 코드로 답해 봅니다.

## 0. 준비: 데이터 불러오기 (5-1과 같음)

In [ ]:
!pip -q install koreanize-matplotlib
import koreanize_matplotlib
import pandas as pd

DATA = 'https://raw.githubusercontent.com/Daeil-Noh/Lectures/main/ax-microdegree/ai-data-analysis/data/'
df = pd.read_csv(DATA + 'process.csv')
print(df.shape)

## 1. 열 고르기
- 대괄호 하나 + 열 이름 → **Series** (열 하나)
- 대괄호 안에 열 이름 **리스트** → **DataFrame** (여러 열)

In [ ]:
temp = df['temp']                       # Series
print(type(temp), temp.shape)
print('평균 온도:', round(temp.mean(), 2))

two = df[['temp', 'pressure']]          # DataFrame
print(type(two), two.shape)
two.head(3)

## 2. 조건 = 행마다 붙는 True/False
`df['temp'] > 220` 은 행마다 True/False를 돌려줍니다. 이것을 `df[ ]` 안에 넣으면 **True인 행만** 남습니다.

In [ ]:
mask = df['temp'] > 220        # True/False 494개
print(mask.head())

hot = df[mask]                 # True인 행만
print('220℃ 초과 제품:', len(hot), '개')
hot[['product_id', 'temp', 'defect_type']].head()

### 질문 1. “온도가 220℃를 넘은 제품은 전체의 몇 %이고, 그 제품들의 불량률은?”
True는 1, False는 0으로 계산됩니다. 그래서 `sum()` = 개수, `mean()` = 비율입니다.

In [ ]:
print('개수:', mask.sum())
print('비율:', round(mask.mean() * 100, 1), '%')
print('220℃ 초과 제품의 불량률:', round(hot['defect'].mean() * 100, 1), '%')
print('전체 불량률:', round(df['defect'].mean() * 100, 1), '%')

### 질문 2. “야간에 220℃를 넘은 제품은 몇 개?”
조건 두 개는 **각각 괄호로 감싸고** `&`(그리고) 또는 `|`(또는)로 묶습니다. 파이썬의 `and`·`or`는 쓸 수 없습니다.

In [ ]:
night_hot = df[(df['temp'] > 220) & (df['shift'] == '야간')]
print('야간 그리고 220℃ 초과:', len(night_hot))

either = df[(df['temp'] > 220) | (df['shift'] == '야간')]
print('야간 또는 220℃ 초과:', len(either))

### 질문 3. “찍힘·스크래치 불량만 보여 줘. 어느 라인에서 많이 나왔지?”
여러 값 중 하나에 해당하는지는 `isin([값 목록])` 으로 고릅니다.

In [ ]:
surface = df[df['defect_type'].isin(['찍힘', '스크래치'])]
print('찍힘·스크래치:', len(surface), '개')
surface['line'].value_counts()

### 질문 4. “인장강도가 가장 낮은 제품 5개는? 압력은 얼마였지?”
`sort_values('열')`은 오름차순(작은 값부터), `ascending=False`는 내림차순입니다.
`nsmallest(N, '열')` / `nlargest(N, '열')`은 정렬 + 위에서 N개를 한 번에 합니다.

In [ ]:
cols = ['product_id', 'line', 'pressure', 'strength', 'defect_type']
df.sort_values('strength').head(5)[cols]

In [ ]:
# 같은 결과를 한 줄로
df.nsmallest(5, 'strength')[cols]

### 질문 5. “온도가 가장 높았던 제품 5개는 어느 라인·호기였지?”

In [ ]:
df.nlargest(5, 'temp')[['product_id', 'line', 'machine', 'temp', 'defect_type']]

## 3. 검증 습관: 다른 방법으로 다시 세기
조건 코드를 AI가 만들었든 내가 만들었든, **경계값**(`>` vs `>=`)과 **개수**를 꼭 확인합니다.

In [ ]:
a = len(df[(df['temp'] > 220) & (df['shift'] == '야간')])
b = ((df['temp'] > 220) & (df['shift'] == '야간')).sum()
c = len(df[(df['temp'] >= 220) & (df['shift'] == '야간')])   # 이상(>=)이면?
print('초과(>) 두 가지 방법:', a, b)
print('이상(>=):', c, '← 정확히 220.0℃인 제품이 섞이면 달라진다')
print('결과의 최저 온도:', night_hot['temp'].min(), '/ 근무조 종류:', night_hot['shift'].unique())

## 4. 슬라이드의 12행 표본
슬라이드 “필터 만들기”, “줄 세우기”에서 쓴 12행은 아래 코드로 똑같이 뽑을 수 있습니다(`random_state`를 고정하면 매번 같은 행).

In [ ]:
sample = df.sample(12, random_state=16)
sample[['product_id', 'shift', 'line', 'temp', 'pressure', 'thickness', 'strength', 'defect_type']]

## 5. 직접 바꿔 보기
1. `TH`(온도 기준)를 210, 215, 225로 바꿔 보세요. 개수와 불량률이 어떻게 달라지나요?
2. `SHIFT`를 `'주간'`으로 바꿔 보세요.
3. `LINES`를 `['3라인']`처럼 바꾸고, `KEY`를 `'thickness'`로 바꿔 두께가 가장 얇은 제품을 찾아보세요.

In [ ]:
TH = 220                    # 1) 온도 기준 (초과)
SHIFT = '야간'              # 2) '주간' 또는 '야간'
LINES = ['1라인', '2라인']  # 3) 고를 라인 목록
KEY = 'strength'            # 3) 정렬 기준 열

sel = df[(df['temp'] > TH) & (df['shift'] == SHIFT) & (df['line'].isin(LINES))]
print('조건에 맞는 제품:', len(sel), '개')
if len(sel) > 0:
    print('그중 불량률:', round(sel['defect'].mean() * 100, 1), '%')
sel.nsmallest(3, KEY)[['product_id', 'line', 'temp', KEY, 'defect_type']]

## 정리
- `df['a']` → Series, `df[['a', 'b']]` → DataFrame
- `df[조건]` → True인 행만. 조건 여러 개는 **괄호 + `&` / `|`**, 여러 값 중 하나는 `isin`
- 골랐으면 `len`·`sum`·`mean`으로 **세어 보고**, 경계값(`>` vs `>=`)을 점검
- `sort_values`(오름차순 기본, `ascending=False`), `nlargest`·`nsmallest`

**생성형AI에게 물어볼 만한 질문**
- “pandas DataFrame `df`에 열 temp(실수), shift('주간'/'야간'), line('1라인'~'3라인')이 있어. 야간에 온도가 215℃ 초과인 1·2라인 제품만 골라 strength가 낮은 순서로 5개 보여 주는 코드를 써 줘.”
- “이 코드에서 `&` 대신 `and`를 쓰면 왜 오류가 나는지 초보자에게 설명해 줘.”

AI가 준 코드는 **결과 개수를 다른 방법으로 다시 세어** 확인하고, 조건의 경계(초과/이상)가 질문과 같은지 확인하세요.